# Double well

### IMPORTS

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from matplotlib.ticker import MultipleLocator

np.random.seed(42)
print('numpy:', np.__version__)

numpy: 2.4.2


### Pauli matrices and the cost hamilton

In [5]:
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

II = np.kron(I2, I2)
ZZ = np.kron(Z, Z)

H_C = -0.5 * (II - ZZ)

print('H_C diagonal {|00>, |01>, |10>, |11>}')
print(' ', np.diag(H_C).real)
print()

basis = {
    '|00>' : np.array([1, 0, 0, 0], dtype=complex),
    '|01>' : np.array([0, 1, 0, 0], dtype=complex),
    '|10>' : np.array([0, 0, 1, 0], dtype=complex),
    '|11>' : np.array([0, 0, 0, 1], dtype=complex)
}

spin_map = {
    '|00>' : '|up-up>',
    '|01>' : '|up-dn>',
    '|10>' : '|dn-up>',
    '|11>' : '|dn-dn>'
}

for lbl, v in basis.items():
    ev = np.real(v.conj().T @ H_C @ v)
    tag = " <- MAX-CUT solution (cut state)" if ev < 0 else ""
    print(f"<{lbl}|H_C|{lbl}> = {ev:+.1f} {spin_map[lbl]}{tag}")

J = 1.0
H_ex = (J/4) * (np.kron(X,X) + np.kron(Y,Y) + np.kron(Z,Z))
vals_ex = np.sort(np.linalg.eigvalsh(H_ex))

print(f'\nHeisenberg H_ex eigenvalues (J={J}):', vals_ex)
print(f'\nSinglet E = - 3J/4 = {-3*J/4}, triplet E = J/4 = {J/4}')


H_C diagonal {|00>, |01>, |10>, |11>}
  [-0. -1. -1. -0.]

<|00>|H_C||00>> = +0.0 |up-up>
<|01>|H_C||01>> = -1.0 |up-dn> <- MAX-CUT solution (cut state)
<|10>|H_C||10>> = -1.0 |dn-up> <- MAX-CUT solution (cut state)
<|11>|H_C||11>> = +0.0 |dn-dn>

Heisenberg H_ex eigenvalues (J=1.0): [-0.75  0.25  0.25  0.25]

Singlet E = - 3J/4 = -0.75, triplet E = J/4 = 0.25


### QAOA Unitaries

In [6]:
def cost_unitary(gamma):
    phases = np.exp(-1j * gamma * np.diag(H_C).real)
    return np.diag(phases)

def mixer_unitary(beta):
    c, s = np.cos(beta), np.sin(beta)

    # Single-qubit mixer: exp(-i beta X)
    Ux = np.array([
        [c, -1j * s],
        [-1j * s, c]
    ], dtype=complex)

    # Two-qubit mixer: exp(-i beta X) ⊗ exp(-i beta X)
    return np.kron(Ux, Ux)


Uc = cost_unitary(1.2)
Ub = mixer_unitary(0.4)

print(f'U_C unitary error: {np.max(np.abs(Uc.conj().T @ Uc - II)):.2e}')
print(f'U_B unitary error: {np.max(np.abs(Ub.conj().T @ Ub - II)):.2e}')

print(f'\nU_C(pi/2) diagonal:')

Uc_test = cost_unitary(np.pi / 2)

for lbl, ph in zip(['|00>', '|01>', '|10>', '|11>'], np.diag(Uc_test)):
    print(f'{lbl}: {ph:.4f}')

U_C unitary error: 0.00e+00
U_B unitary error: 0.00e+00

U_C(pi/2) diagonal:
|00>: 1.0000+0.0000j
|01>: 0.0000+1.0000j
|10>: 0.0000+1.0000j
|11>: 1.0000+0.0000j


### QAOA statevector and expected cost

In [7]:
def qaoa_state(beta, gamma):
    plus = np.ones(4, dtype=complex) / 2.0
    return mixer_unitary(beta) @ cost_unitary(gamma) @ plus

def expected_cost(beta, gamma):
    psi = qaoa_state(beta, gamma)
    return float(np.real(psi.conj() @ H_C @ psi))

def closed_form(beta, gamma):
    return -0.5 + 0.5 * np.sin(4* beta) * np.sin(gamma)

N = 80
betas = np.linspace(0, np.pi/2, N)
gammas = np.linspace(0, 2*np.pi, N)
B, G = np.meshgrid(betas, gammas, indexing='ij')

F_num = np.array([[expected_cost(b, g) for g in gammas] for b in betas])
F_ana = closed_form(B, G)

print(f'Max pointwise error (statevector vs closed form): {np.max(np.abs(F_num - F_ana)):.2e}')
print(f'F range [num]:  [{F_num.min():.4f}, {F_num.max():.4f}]')
print(f'F range [ana]:  [{F_ana.min():.4f}, {F_ana.max():.4f}]')

Max pointwise error (statevector vs closed form): 6.66e-16
F range [num]:  [-0.9998, -0.0002]
F range [ana]:  [-0.9998, -0.0002]
